# U2NormalNodeBehaviorFrp — R0011 新绘图执行候选

**当前不是已执行成功的工作2成果。** 本Pro环境没有 mquant/Uplot；下列新代码须在现场 Jupyter 正常执行、检查交互，并由原专业Pro判断。不得加载旧图恢复资格。代码按具名正式导出重新构建 Uplot/CAEGraph，不读取旧 HTML 或截图。

同目录 `DISPLAY_PLAN.json` 是本次实际采用的可修订场景配置。已有图元 helper 只作未验证参考。Notebook 内的来源读取是既有 OnlyRead 导出的核源复用，**不是此刻重新运行 Feature**。稀疏知识切点只画实际点，不虚构期间保持。

在具备现场依赖的 Jupyter kernel 中打开此文件，确认本地 observe-iterate 仓库含两个锁定提交；必要时设置 `OBSERVE_REPO` 与 `BI01_R11_ARTIFACT_DIR` 为实际目录。Restart Kernel → Run All，再切换已取 K、hover、缩放/复位。GitHub只存文件，不保证运行 JavaScript。


In [1]:
from importlib.util import find_spec
import json
needed = ["mquant", "ipywidgets"]
missing = [name for name in needed if find_spec(name) is None]
print(json.dumps({"required": needed, "missing": missing}, ensure_ascii=False))
if missing:
    raise RuntimeError("现场绘图依赖未就绪：" + ", ".join(missing) + "；未执行Uplot，不能标记Notebook完成。")


{"required": ["mquant", "ipywidgets"], "missing": ["mquant", "ipywidgets"]}


RuntimeError: 现场绘图依赖未就绪：mquant, ipywidgets；未执行Uplot，不能标记Notebook完成。

## 正式来源与运行版本
仅读取本地 Git 中的指定不可变版本；不访问网络、不重算 N、不修改共享源码。完整生产配置沿原 R30，下面的观察范围不是生产参数。运行时应保留实际导入路径、HEAD及影响运行的既有未提交差异；不能仅凭HEAD声称干净锁定。


In [ ]:
from pathlib import Path
from functools import lru_cache
import io, os, subprocess, inspect, hashlib
import pandas as pd
import mquant.uplot as uplot
import mquant.aidata.u2_normal_node_behavior_display as nd
from mquant.quant import OptContext
from bokeh.io import output_notebook, show
from bokeh.resources import INLINE
from IPython.display import display, clear_output
import ipywidgets as widgets

ARTIFACT_DIR = Path(os.environ.get("BI01_R11_ARTIFACT_DIR", ".")).resolve()
PLAN = json.loads((ARTIFACT_DIR / "DISPLAY_PLAN.json").read_text(encoding="utf-8"))
REPO = Path(os.environ.get("OBSERVE_REPO", "/dwdhome/fy/observe-iterate-bi01-fo-01")).resolve()
if not REPO.is_dir():
    raise FileNotFoundError(f"实际仓库不存在：{REPO}")
W02 = "runs/BI-01-FO-01/artifacts/worker-BI01-FO01-N-DISPLAY-INTEGRATION-02/"
W03 = "runs/BI-01-FO-01/artifacts/worker-BI01-FO01-N-READABILITY-CROSSK-03/"
SOURCE_READS = {}
@lru_cache(maxsize=160)
def source_bytes(ref, path):
    result = subprocess.run(["git", "-C", str(REPO), "show", f"{ref}:{path}"], capture_output=True, check=True)
    raw = result.stdout
    SOURCE_READS[f"{ref}:{path}"] = hashlib.sha256(raw).hexdigest()
    return raw

def frame_at(ref, path, formal=True):
    frame = pd.read_csv(io.BytesIO(source_bytes(ref, path)), dtype="string")
    time_col = "timestamp" if formal else frame.columns[0]
    frame[time_col] = pd.to_datetime(frame[time_col])
    for col in frame.columns:
        if col == time_col:
            continue
        if col == "text" or col.endswith("_text"):
            frame[col] = frame[col].fillna("")
        elif col.endswith("_high") or col.endswith("_low"):
            frame[col] = pd.array(frame[col], dtype="UInt64")
        else:
            try:
                frame[col] = pd.to_numeric(frame[col], errors="raise")
            except (ValueError, TypeError):
                pass
    if not formal:
        return frame.set_index(time_col)
    children = sorted([c for c in frame if c.startswith("child_id_")], key=lambda c: int(c.rsplit("_", 1)[1]))
    return frame.set_index(["timestamp", "occurrence", "record_id", *children])

def cut_tables(ref, folder):
    return {name: frame_at(ref, folder + name + ".csv") for name in nd.FORMAL_PRODUCTS}

def merge_real_cuts(cuts):
    result = {}
    for key in nd.FORMAL_PRODUCTS:
        frames = [tables[key] for tables in cuts.values()]
        names = list(frames[0].index.names)
        merged = pd.concat(frames).reset_index().drop_duplicates()
        if merged.duplicated(names).any():
            raise ValueError(f"同原表坐标内容冲突：{key}；不得按最新值覆盖")
        result[key] = merged.set_index(names).sort_index()
    return result

context = OptContext(sof_name=PLAN["sof_name"])
output_notebook(resources=INLINE, hide_banner=True)
for module in (uplot, nd):
    path = Path(inspect.getfile(module)).resolve()
    print(module.__name__, str(path), hashlib.sha256(path.read_bytes()).hexdigest())


## rb：知识切点演变与回看
第一幅只比较三个已取切点，不等同连续433发生。随后通过选择器重新构建选定K图；可观察确认前/后成员及working变化。重新运行同一个K也重新调用Uplot，不从历史页面载入图。


In [ ]:
def new_selected(tables, price, symbol, K):
    bundle = nd.build_selected_cut_bundle(tables, price, symbol=symbol,
        color=PLAN["instrument_colors"][symbol], as_of=int(K),
        optional_panels=tuple(PLAN["optional_panels"]))
    assert int(bundle.meta["resolved_origin_as_of"]) <= K
    assert int(bundle.price["open_uti"].max()) <= K
    bundle.meta["formal_entry"] = "Pinned normal-OnlyRead export reuse; not a fresh Feature read"
    bundle.meta["display_identity"]["r0011_plan"] = PLAN["version"]
    return bundle

def sparse_evolution(cuts, selected, price, symbol, Ks):
    bundle = nd.build_evolution_bundle(merge_real_cuts(cuts), price, symbol=symbol,
        color=PLAN["instrument_colors"][symbol], start_uti=min(Ks), end_uti=max(Ks),
        optional_panels=tuple(PLAN["optional_panels"]))
    for name in bundle.views:
        if name.endswith("_hold") or name.endswith("_steps"):
            bundle.views[name] = bundle.views[name].iloc[0:0].copy()
        elif name.endswith("_updates") or name.endswith("_points"):
            bundle.views[name] = pd.concat([b.views[name] for b in selected.values()], ignore_index=True)
    bundle.meta["formal_entry"] = "Sparse pinned knowledge-cut exports; NOT continuous occurrence history"
    bundle.meta["display_identity"]["r0011_plan"] = PLAN["version"]
    bundle.meta["display_identity"]["observed_Ks"] = sorted(Ks)
    return bundle

rb_cuts = {name: cut_tables(PLAN["source_refs"]["W02"], W02 + "data/selected_cuts/" + name + "/")
           for name in PLAN["rb_cuts"]}
rb_price = frame_at(PLAN["source_refs"]["W02"], W02 + "data/same_bound_price/rb_5m_price_calendar.csv", False)
rb_selected = {name: new_selected(rb_cuts[name], rb_price, "rb", K) for name, K in PLAN["rb_cuts"].items()}
rb_sparse = sparse_evolution(rb_cuts, rb_selected, rb_price, "rb", list(PLAN["rb_cuts"].values()))
show(nd.build_uplot_layout(rb_sparse, context=context, calendar_name=PLAN["calendar"],
    title_suffix="R0011 新绘制：离散知识切点；不推断期间保持"))


In [ ]:
rb_K = widgets.Dropdown(options=[(name, name) for name in PLAN["rb_cuts"]], value="final", description="知识切点")
rb_out = widgets.Output()
def display_rb_cut(change=None):
    name = rb_K.value
    with rb_out:
        clear_output(wait=True)
        bundle = new_selected(rb_cuts[name], rb_price, "rb", PLAN["rb_cuts"][name])
        show(nd.build_uplot_layout(bundle, context=context, calendar_name=PLAN["calendar"],
            title_suffix="R0011 新绘制：" + name))
rb_K.observe(display_rb_cut, names="value")
display(rb_K, rb_out)
display_rb_cut()


## i：缺价后部分恢复
三个知识切点同属原working。预期数据问题是 progress 恢复而 efficiency/counter仍缺，waiting=0而ratio缺分母。下面同时显示原字段状态；图形/标签若未充分表达局部重启与真实零，应在工作2修订直接消费者，再交Pro判断，不能仅凭表格宣布画法通过。


In [ ]:
i_cuts = {name: cut_tables(PLAN["source_refs"]["W03"], W03 + "data/crossk/" + name + "/")
          for name in PLAN["i_cuts"]}
i_price = frame_at(PLAN["source_refs"]["W02"], W02 + "data/i_missing_context/i_5m_price_calendar.csv", False)
i_selected = {name: new_selected(i_cuts[name], i_price, "i", K) for name, K in PLAN["i_cuts"].items()}
i_sparse = sparse_evolution(i_cuts, i_selected, i_price, "i", list(PLAN["i_cuts"].values()))
show(nd.build_uplot_layout(i_sparse, context=context, calendar_name=PLAN["calendar"],
    title_suffix="R0011 新绘制：末缺价、恢复、下一拍；仅三处实得状态"))
fields = ("working_progress", "working_efficiency", "working_counter", "working_duration", "working_waiting", "working_waiting_ratio")
status_rows = []
for label, tables in i_cuts.items():
    for field in fields:
        for row in tables[field].reset_index().to_dict("records"):
            status_rows.append({"cut": label, "K": PLAN["i_cuts"][label], "field": field,
                **{key: row.get(key) for key in ("value", "present", "knownness", "applicability", "reason")}})
display(pd.DataFrame(status_rows))
# These original fields are audit support. They do not substitute for checking
# whether the NEW graphic makes the missing/zero/local-restart distinction clear.


## hc：正式分母及价格核对（此处是数据辅助）
不为每个纯数据问题强制画图。原表读出与参考计算分开；这里用完整已导出行情作可重复参考，不能把结果写回正式N。

## 当前尚需完成的真实验证
新Notebook各图实际执行/交互及专业判断；必要信息/状态的可读表达；aidata三公开入口、平台页面实际打开、同run/node正式数据及后续分析。此Notebook不调用自动Root/计时器，也不代替平台交付。全部旧图不采用，技术代码的可借用性不改变这一点。


In [ ]:
hc_ref = PLAN["source_refs"]["W03"]
hc_folder = W03 + "data/crossk/hc_final/"
hc_price = frame_at(hc_ref, hc_folder + "hc_5m_price_calendar.csv", False)
hc_rhythm = frame_at(hc_ref, hc_folder + "rhythm.csv").reset_index()
hc_intervals = frame_at(hc_ref, hc_folder + "intervals.csv").reset_index().sort_values("child_id_3")
assert hc_intervals["present"].eq(1).all()
denominator = float(hc_intervals.tail(3)["duration"].median())
assert denominator == float(hc_rhythm.iloc[0]["current_median_value"])
K = int(hc_price["open_uti"].max())
last_update = int(hc_rhythm.iloc[0]["last_update_value"])
print({"formal_recent_intervals": hc_intervals.tail(3)["duration"].tolist(),
       "formal_median": denominator, "waiting_reference": K-last_update,
       "ratio_reference": (K-last_update)/denominator,
       "qualification": "reference arithmetic beside formal tables; no new production N"})
print("实际核源文件数", len(SOURCE_READS))
